Vision Transformer(ViT)는 이미지 인식 문제를 기존의 CNN 방식이 아니라 Transformer 구조로 직접 해결한 모델입니다. ViT는 이미지를 작은 패치(patch) 단위로 나눈 뒤 이를 토큰(token)처럼 취급하고, 각 패치 간의 관계를 Self-Attention으로 학습함으로써 이미지 전체의 전역적 문맥을 효과적으로 이해합니다. 이 접근은 국소적인 특징 추출에 강한 CNN과 달리, 멀리 떨어진 영역 간의 상관관계를 한 번에 파악할 수 있다는 장점이 있으며, 충분한 데이터와 함께 학습될 경우 CNN 기반 모델을 능가하는 성능을 보입니다. 결과적으로 ViT는 “이미지를 시퀀스로 바라본다”는 관점 전환을 통해 컴퓨터 비전과 자연어 처리의 경계를 허문 대표적인 모델로 평가받고 있습니다.

<img src='https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdna%2FbFc5ve%2FdJMcaioziMq%2FAAAAAAAAAAAAAAAAAAAAAHMR5IOZIG-mScOk6o4qUYdXuXgeWHcnbg9bAg7YSBZQ%2Fimg.png%3Fcredential%3DyqXZFxpELC7KVnFOS48ylbz2pIh7yKj8%26expires%3D1772290799%26allow_ip%3D%26allow_referer%3D%26signature%3DOmX6SR6ZmrwePiqH0Hzj6BQUdwg%253D'>

###① Input Image
모델에 입력되는 원본 이미지입니다. ViT는 입력 이미지를 그대로 합성곱 연산에 적용하지 않고, 이후 단계에서 일정한 크기의 패치 단위로 분할하여 처리합니다.



###② Divide Image into Patches
입력 이미지를 고정 크기의 패치(patch)로 균등하게 분할합니다.
이 과정은 ViT의 핵심 전처리 단계로, 이미지 전체를 패치들의 시퀀스(sequence)로 재구성하기 위한 준비 단계입니다.



###③ Image Patches
분할된 각 패치는 이미지의 국소적인 시각 정보(local visual information)를 포함하며, 이후 독립적인 처리 단위로 사용됩니다. 이 시점에서 이미지는 더 이상 하나의 2차원 행렬이 아니라, 여러 개의 패치 집합으로 표현됩니다.



###④ Convert Patches into Tokens
각 이미지 패치는 펼침(flatten) 연산을 거친 후 선형 변환을 통해 고정 차원의 벡터(embedding)로 변환됩니다. 이렇게 변환된 벡터는 Transformer에 입력되는 Patch Token이 되며, 이는 자연어 처리에서의 토큰과 동일하게 모델의 기본 입력 단위로 취급됩니다.



###⑤ Transformer Encoder
Patch Token 시퀀스는 Transformer Encoder에 입력됩니다.
Encoder 내부에서는

- Self-Attention을 통해 모든 패치 간의 상호 관계를 계산하고
- Feed Forward Network를 통해 특징을 비선형적으로 변환하며
- Residual Connection과 Layer Normalization을 통해 학습 안정성과 정보 보존을 유지합니다.

이 구조는 입력 토큰의 순차 처리 없이 병렬적으로 계산이 이루어진다는 특징을 가집니다.



###⑥ Learning Global Context
Self-Attention 메커니즘을 통해 ViT는 이미지 내의 패치 간 관계를 전역적(global)으로 학습합니다. 이로 인해 공간적으로 멀리 떨어진 영역 간의 연관성도 직접적으로 모델링할 수 있으며, 이는 국소 수용 영역에 의존하는 CNN과 구별되는 중요한 특성입니다.



###⑦ Output (Classification Result)
Transformer Encoder의 출력은 분류 헤드(classification head)를 거쳐 최종 예측 결과를 생성합니다. 이 결과는 이미지 전체의 패치 정보를 종합적으로 반영한 판단에 기반합니다.





**※ Vision Transformer는 이미지를 패치 단위의 시퀀스로 변환한 뒤, Transformer의 Self-Attention 구조를 활용하여 이미지 전역의 관계를 학습하는 비전 모델입니다.**

 - 트랜스포머 모델이 데이터가 늘어남에 따라 성능의 향상이 기존 CNN모델보다 좋음

 - 학습 데이터가 많아질수록 트랜스포머가 CNN의 성능을 능가

 - 이미지 크기가 달라짐에 따라, patch를 변경하는 등 유연하게 대응 가능

 - 이미지가 커지면 patch 수의 증가로 조절

<img src='https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdna%2FqmDly%2FdJMb99SJhTK%2FAAAAAAAAAAAAAAAAAAAAAGTwgwT9OBFc2gNMxDrVHeUGmrortPK99qBbjD003ozS%2Fimg.png%3Fcredential%3DyqXZFxpELC7KVnFOS48ylbz2pIh7yKj8%26expires%3D1772290799%26allow_ip%3D%26allow_referer%3D%26signature%3Dz0tdlmxUPe2AD6JtSK6ajJp93U8%253D'>

In [15]:
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

image = np.random.rand(8, 8, 3).astype(np.float32)

image = image.reshape(2, 4, 2, 4, 3)
image = image.transpose(0, 2, 1, 3, 4)
patches = image.reshape(-1, 4, 4, 3) #(4, 4, 4, 3)

print("patches:", patches.shape)

patches = torch.tensor(patches)

embedding_dim = 32
num_heads = 4
num_transformer_layers = 2
num_classes = 10
mlp_dim = 256

def patch_embedding(patches, embedding_dim):
    N, Ph, Pw, C = patches.shape
    patch_dim = Ph * Pw * C

    patch_flat = patches.reshape(N, patch_dim) #(4, 48)

    patch_flat = patch_flat.unsqueeze(0)

    proj = nn.Linear(patch_dim, embedding_dim) #(48, 32)
    tokens = proj(patch_flat)

    return tokens, proj

tokens, patch_proj = patch_embedding(patches, embedding_dim)
print("tokens:", tokens.shape)

B, N, D = tokens.shape

cls_token = nn.Parameter(torch.zeros(1, 1, D))
x = torch.cat([cls_token.expand(B, -1, -1), tokens], dim=1)

pos_embed = nn.Parameter(torch.randn(1, x.size(1), D) * 0.02)
x = x + pos_embed

encoder_layer = nn.TransformerEncoderLayer(
    d_model=D,
    nhead=num_heads,
    dim_feedforward=mlp_dim,
    batch_first=True
)
transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_transformer_layers)

z = transformer_encoder(x)

classification_layer = nn.Linear(D, num_classes)
logits = classification_layer(z[:, 0, :])
output = F.log_softmax(logits, dim=-1)

pred_class = output.topk(1, dim=-1).indices.item()
print("class :", pred_class)

patches: (4, 4, 4, 3)
tokens: torch.Size([1, 4, 32])
class : 6


In [16]:
# 번호판 인식 ViT 모델

In [17]:
!mkdir -p "/content/drive/MyDrive/AI 활용 멀티모달 MCP PJ 시즌1/4. 멀티모달/data/ViT"

In [ ]:
!unzip -q "/content/drive/MyDrive/AI 활용 멀티모달 MCP PJ 시즌1/4. 멀티모달/data/ViT.zip" -d "/content/drive/MyDrive/AI 활용 멀티모달 MCP PJ 시즌1/4. 멀티모달/data/ViT"

replace /content/drive/MyDrive/AI 활용 멀티모달 MCP PJ 시즌1/4. 멀티모달/data/ViT/annotations.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import json

In [ ]:
data_root = "/content/drive/MyDrive/AI 활용 멀티모달 MCP PJ 시즌1/4. 멀티모달/data/ViT"
annotation_filename = 'annotations.json'
image_dir = os.path.join(data_root, 'images')

In [ ]:
with open(os.path.join(data_root, annotation_filename), 'r') as json_f:
    annotations = json.load(json_f)

In [ ]:
annotations[:3]

In [ ]:
len(annotations)

In [ ]:
import json
from collections import defaultdict

In [ ]:
digit1_cnt = defaultdict(int)
digit2_cnt = defaultdict(int)
digit3_cnt = defaultdict(int)
digit4_cnt = defaultdict(int)

for annot in annotations:
    labels = annot['labels']

    digit1_cnt[labels[0]] += 1
    digit2_cnt[labels[1]] += 1
    digit3_cnt[labels[2]] += 1
    digit4_cnt[labels[3]] += 1

In [ ]:
with open('digit1_cnt.json', 'w') as json_f:
    json.dump(digit1_cnt, json_f, indent='\t', ensure_ascii=False)
with open('digit2_cnt.json', 'w') as json_f:
    json.dump(digit2_cnt, json_f, indent='\t', ensure_ascii=False)
with open('digit3_cnt.json', 'w') as json_f:
    json.dump(digit3_cnt, json_f, indent='\t', ensure_ascii=False)
with open('digit4_cnt.json', 'w') as json_f:
    json.dump(digit4_cnt, json_f, indent='\t', ensure_ascii=False)

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from PIL import Image
import numpy as np

In [ ]:
class JsonDataset(Dataset):
    def __init__(self, image_dir, annotations, transform=None):
        self.image_dir = image_dir
        self.annotations = annotations
        self.transform = transform
        self.class_list = [i for i in range(10)]
        self.num_classes = len(self.class_list)

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        annot = self.annotations[idx]
        filename = annot['filename']
        classes = annot['labels']
        image_path = os.path.join(self.image_dir, filename)
        image = Image.open(image_path).convert('RGB')
        target_list = []
        for cls in classes:
            target_list.append(int(cls))
        target_list = torch.Tensor(target_list).to(torch.long)
        if self.transform:
            image = self.transform(image=np.array(image))['image']
        return image, target_list

In [ ]:
dataset = JsonDataset(image_dir = image_dir, annotations = annotations)
data = dataset[0]
data

In [ ]:
data[0]

In [ ]:
data[1]

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
def draw_images(images, classes):
    fig, axs = plt.subplots(2, 4, figsize = (12, 6))
    for i, ax in enumerate(axs.flat):
        ax.imshow(images[i])
        ax.set_title(classes[i])
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
import random

In [ ]:
random.shuffle(annotations)
sample_images =[]
sample_classes = []
sample_cnt = 0
max_cnt = 8

In [ ]:
for annot in annotations:
    sample_classes.append(annot['labels'])
    image_path = os.path.join(image_dir, annot['filename'])
    image = Image.open(image_path).convert('RGB')
    sample_images.append(np.array(image))
    sample_cnt += 1
    if sample_cnt == max_cnt:
        break

In [ ]:
draw_images(sample_images, sample_classes)

In [ ]:
sample_classes

In [ ]:
len_annot = len(annotations)
train_annot = annotations[:int(len_annot * 0.9)]
val_annot = annotations[int(len_annot * 0.9) : ]

len(train_annot), len(val_annot)

In [ ]:
hyper_params = {
    'num_epochs': 3,
    'lr': 0.0001,
    'image_size': 224,
    'train_batch_size': 32,
    'val_batch_size': 16,
    'print_preq': 0.1
}

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [ ]:
sample_transform = A.Compose([
    # rotate_limit: -15 ~ 15 사이로 랜덤 회전
    # shift_limit: 5% 이내로 상하좌우 이동
    # scale_limit: -10 ~ 10 사이로 확대 / 축소
    # p: 50% 확률로 적용
    # border_mode: 빈 공간은 검은색으로 채움
    A.ShiftScaleRotate(rotate_limit=15, shift_limit=0.05, scale_limit=0.1, p=0.5, border_mode=0),
    # 이미지의 가로 / 세로 중 긴 쪽으로 기준으로 비율을 유지한 채 max_size로 축소 또는 확대
    A.LongestMaxSize(max_size=hyper_params['image_size']),
    # 한쪽 변이 부족한 영역이 있다면 패딩으로 채움
    A.PadIfNeeded(min_height=hyper_params['image_size'], min_width=hyper_params['image_size'], border_mode=0)
])

In [ ]:
sample_dataset = JsonDataset(image_dir=image_dir, annotations=annotations, transform=sample_transform)
sample_dataset[3][0]

In [ ]:
transformed_images = []
targets = []

max_cnt = 8
for idx, (image, target) in enumerate(sample_dataset):
    if idx == max_cnt:
        break
    transformed_images.append(image)
    targets.append(target.tolist())

In [ ]:
target_classes = []
class_list = sample_dataset.class_list

for target in targets:
    classes = []
    for cls in target:
        classes.append(class_list[cls])
    target_classes.append(classes)

target_classes

In [ ]:
draw_images(transformed_images, target_classes)

In [ ]:
train_transform = A.Compose([
    A.ShiftScaleRotate(rotate_limit=15, shift_limit=0.05, scale_limit=0.1, p=0.5, border_mode=0),
    A.LongestMaxSize(max_size=hyper_params['image_size']),
    A.PadIfNeeded(min_height=hyper_params['image_size'], min_width=hyper_params['image_size'], border_mode=0),
    A.Normalize(p=1.0, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    # HWC -> CHW 차원 변경
    # Numpy -> Tensor 변환
    # 값 범위를 그대로 유지 (0 ~ 255 유지)
    ToTensorV2()
])

val_transform = A.Compose([
    A.LongestMaxSize(max_size=hyper_params['image_size']),
    A.PadIfNeeded(min_height=hyper_params['image_size'], min_width=hyper_params['image_size'], border_mode=0),
    A.Normalize(p=1.0, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [ ]:
train_dataset = JsonDataset(image_dir=image_dir, annotations=train_annot, transform=train_transform)
val_dataset = JsonDataset(image_dir=image_dir, annotations=val_annot, transform=val_transform)

train_dataloader = torch.utils.data.DataLoader(train_dataset, num_workers=4,
                                               batch_size=hyper_params['train_batch_size'], shuffle=True)
val_dataloader = torch.utils.data.DataLoader(val_dataset, num_workers=4,
                                               batch_size=hyper_params['val_batch_size'], shuffle=True)

In [ ]:
# timm(Torch Image Models): PyTorch에서 표준처럼 쓰이는 비전 모델 라이브러리
# 공식 Pytorch 모델은 아니지만 연구+실무 분야에서 공식처럼 사용됨
import timm
import torch.nn as nn

In [ ]:
class ViTMultiBranchClassifier(nn.Module):
    def __init__(self, num_classes, num_branches, model_name='vit_base_patch16_224', pretrained=True):
        super(ViTMultiBranchClassifier, self).__init__()
        self.vit = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        # 여러 개의 모듈을 리스트로 저장
        # 모델에 모듈을 등록하기 위함
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.vit.embed_dim, 256),
                nn.GELU(),
                nn.Dropout(0.3),
                nn.Linear(256, num_classes)
            )
            # 동일한 구조의 분류기(head)를 num_branches개 생성
            # num_branches=3 -> self.branches[0], self.branches[1], self.branches[2]
            for _ in range(num_branches)
        ])

    def forward(self, x):
        features = self.vit(x)
        outputs = [branch(features) for branch in self.branches]
        return outputs

In [ ]:
train_dataset.num_classes

In [ ]:
num_branches = 4
num_classes = train_dataset.num_classes
model = ViTMultiBranchClassifier(num_classes=num_classes, num_branches=num_branches)
model

In [ ]:
sample_input = torch.randn(1, 3, 224, 224)
model(sample_input)

In [ ]:
model(sample_input)[0]

In [ ]:
def calculate_accuracy(y_pred, y_true):
    _, predicted = torch.max(y_pred, 1)
    correct = (predicted == y_true).float()
    accuracy = correct.sum() / len(correct)
    return accuracy

In [ ]:
import torch.optim as optim

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=hyper_params['lr'])

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

In [ ]:
num_epochs = hyper_params['num_epochs']
model_save_dir = './vit_train_results'
best_acc = 0.0

In [ ]:
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    epoch_loss = 0.0
    print_cnt = int(len(train_dataloader) * hyper_params['print_preq'])
    for idx, (images, targets) in enumerate(train_dataloader):
        images, targets = images.to(device), targets.to(device)
        outputs = model(images)
        total_loss = 0
        for branch_idx, output in enumerate(outputs):
            loss = nn.CrossEntropyLoss()(output, targets[:, branch_idx])
            total_loss += loss
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()

        running_loss += total_loss.item()
        epoch_loss += total_loss.item()

        if(idx > 0) and (idx % print_cnt) == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}], "
                  f"Iter [{idx}/{len(train_dataloader)}] "
                  f"Loss: {(running_loss/print_cnt)/num_branches:.4f}")
                running_loss = 0.0

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(train_dataloader):.4f}")
    accuracies = [[] for _ in range(num_branches)]

    model.eval()
    with torch.no_grad():
        for images, targets in val_dataloader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            for i, output in enumerate(outputs):
                accuracy = calculate_accuracy(output, targets[:, i])
                accuracies[i].append(accuracy.item())
    total_acc = 0.0
    for idx, acc_list in enumerate(accuracies):
        mean_accuracy = sum(acc_list) / len(acc_list)
        total_acc += mean_accuracy
        print(f'Branch {idx + 1} Accuracy: {mean_accuracy*100:.2f}%')

    if total_acc > best_acc:
        os.makedirs(model_save_dir, exist_ok=True)
        model_save_path = os.path.join(model_save_dir, 'best_model.pth')
        torch.save(model.state_dict(), model_save_path)
        best_acc = total_acc

In [ ]:
from torchvision.transforms.functional import to_pil_image

In [ ]:
def denormalize(tensor, mean, std):
    mean = torch.tensor(mean).reshape(-1, 1, 1)
    std = torch.tensor(std).reshape(-1, 1, 1)
    tensor = tensor * std + mean
    tensor = torch.clamp(tensor, 0, 1)
    return tensor

In [ ]:
def tensor_to_pil(tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    tensor = denormalize(tensor, mean, std)
    pil_image = to_pil_image(tensor)
    return pil_image

In [ ]:
model_save_dir = 'vit_train_results'
class_list = [str(i) for i in range(10)]
model = ViTMultiBranchClassifier(num_classes=num_classes, num_branches=num_branches).eval().to(device)
weight = torch.load(os.path.join(model_save_dir, 'best_model.pth'), map_location='cpu')
model.load_state_dict(weight)

In [ ]:
eval_cnt = 8
model.eval()
pred_class_list = []
input_image_list = []

with torch.no_grad():
    for idx, (images, targets) in enumerate(val_dataloader):
        images, targets = images.to(device), targets.to(device)
        outputs = model(images)
        input_image_list.append(tensor_to_pil(images[0].detach().cpu()))
        pred_classes = []
        for i, output in enumerate(outputs):
            output = output[0]
            predicted = torch.argmax(output)
            pred_classes.append(class_list[int(predicted)])
        pred_class_list.append("".join(pred_classes))
        if (idx+1) == eval_cnt:
            break

In [ ]:
pred_class_list

In [ ]:
draw_images(input_image_list, pred_class_list)